In [0]:
notes_clean = spark.read.table("...")

In [0]:
#See Appendix Materials for specific prompts used in analyses
prompt = """...""""

In [0]:
prompt_col = F.format_string(
    prompt,
    F.col("message_text"),
)

notes_with_prompt = notes_clean.withColumn("prompt", prompt_col)

In [0]:
inference_df = notes_with_prompt.withColumn(
    "raw_llm_json",
    F.expr("""
      ai_query(
        'alias',
        prompt,
        map('temperature', 0, 'maxTokens', 300)
      )
    """)
)

In [0]:
#For prompts WITHOUT conditional task structure
response_schema = StructType([
    StructField("statement_of_negative_emotion_pred",   IntegerType()),
    StructField("statement_of_positive_emotion_pred",   IntegerType()),
    StructField("statement_of_progress_pred",  IntegerType()),
    StructField("statement_of_challenge_pred", IntegerType()),
    StructField("explanation",                   StringType())
])

# 2. Parse the cleaned JSON
parsed_df = inference_df.withColumn(
    "parsed",
    F.from_json("raw_llm_json", response_schema)
)

In [0]:
#For prompts WITH conditional task structure
response_schema = StructType([
    StructField("empathic_opportunity_pred",   IntegerType()),
    StructField("statement_of_negative_emotion_pred",   IntegerType()),
    StructField("statement_of_positive_emotion_pred",   IntegerType()),
    StructField("statement_of_progress_pred",  IntegerType()),
    StructField("statement_of_challenge_pred", IntegerType()),
    StructField("explanation",                   StringType())
])

# 2. Parse the cleaned JSON
parsed_df = inference_df.withColumn(
    "parsed",
    F.from_json("raw_llm_json", response_schema)
)

In [0]:
#For prompts with anonymized categories
response_schema = StructType([
    StructField("empathic_opportunity_pred",   IntegerType()),
    StructField("category1",   IntegerType()),
    StructField("category2",   IntegerType()),
    StructField("statement_of_progress_pred",  IntegerType()),
    StructField("statement_of_challenge_pred", IntegerType()),
    StructField("explanation",                   StringType())
])

# 2. Parse the cleaned JSON
parsed_df = inference_df.withColumn(
    "parsed",
    F.from_json("raw_llm_json", response_schema)
)

In [0]:
result_df = parsed_df.select(
    *notes_clean.columns,     # all original fields
    "parsed.*"                # preds + rationale
)

In [0]:
from pyspark.sql.functions import when, col

#For prompts WITHOUT conditional task structure
result_df = (
    result_df
    # empathic_opportunity_pred = 1 if any of the four flags is 1, else 0
    .withColumn(
        "empathic_opportunity_pred",
        when(
            (col("statement_of_negative_emotion_pred") == 1) |
            (col("statement_of_positive_emotion_pred") == 1) |
            (col("statement_of_progress_pred")         == 1) |
            (col("statement_of_challenge_pred")        == 1),
            1
        ).otherwise(0)
    )
    # statement_of_emotion_pred = 1 if negative or positive emotion flag is 1, else 0
    .withColumn(
        "statement_of_emotion_pred",
        when(
            (col("statement_of_negative_emotion_pred") == 1) |
            (col("statement_of_positive_emotion_pred") == 1),
            1
        ).otherwise(0)
    )
)

In [0]:
from pyspark.sql.functions import when, col

#For prompts WITH conditional task structure
result_df = (
    result_df
    .withColumn(
        "statement_of_emotion_pred",
        when(
            (col("statement_of_negative_emotion_pred") == 1) |
            (col("statement_of_positive_emotion_pred") == 1),
            1
        ).otherwise(0)
    )
)

In [0]:
from pyspark.sql.functions import when, col

#For prompts with anonymized categories
result_df = (
    result_df
    .withColumn(
        "statement_of_emotion_pred",
        when(
            (col("category1") == 1) |
            (col("category2") == 1),
            1
        ).otherwise(0)
    )
)

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType
from sklearn.metrics import precision_recall_fscore_support
import pandas as pd

#For prompts without anonymized categories
labels = [
    "empathic_opportunity",
    "statement_of_emotion",
    "valence_negative",
    "valence_positive",
    "statement_of_progress",
    "statement_of_challenge",
]

preds = [
    "empathic_opportunity_pred",
    "statement_of_emotion_pred",
    "statement_of_negative_emotion_pred",
    "statement_of_positive_emotion_pred",
    "statement_of_progress_pred",
    "statement_of_challenge_pred",
]

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType
from sklearn.metrics import precision_recall_fscore_support
import pandas as pd

#For prompts with anonymized categories
labels = [
    "empathic_opportunity",
    "statement_of_emotion",
    "valence_negative",
    "valence_positive",
    "statement_of_progress",
    "statement_of_challenge",
]

preds = [
    "empathic_opportunity_pred",
    "statement_of_emotion_pred",
    "category1",
    "category2",
    "statement_of_progress_pred",
    "statement_of_challenge_pred",
    ]

In [0]:
# Fill in missing predictions with 0s
fill_dict = {c: 0 for c in preds}
df_filled = result_df.na.fill(fill_dict)
df = df_filled

for c in labels + preds:
    df = df.withColumn(c, col(c).cast(IntegerType()))

pdf = df.select(labels + preds).toPandas()

# compute metrics per label
results = {}
for true_col, pred_col in zip(labels, preds):
    p, r, f, _ = precision_recall_fscore_support(
        pdf[true_col],
        pdf[pred_col],
        average="binary",
        zero_division=0
    )
    results[true_col] = {
        "precision": round(p, 4),
        "recall":    round(r, 4),
        "f1_score":  round(f, 4),
    }

metrics_df = pd.DataFrame(results).T
metrics_df.index.name = "label"
print(metrics_df)

In [0]:
# BCa 95% CIs for F1 per category

import numpy as np
import pandas as pd
from pyspark.sql import functions as F

B = 500               # number of bootstrap resamples
ALPHA = 0.05          
RANDOM_SEED = 42

pairs = [
    ("empathic_opportunity",          "empathic_opportunity_pred"),
    ("statement_of_emotion",          "statement_of_emotion_pred"),
    ("valence_negative",              "category1"), #statement_of_negative_emotion_pred
    ("valence_positive",              "category2"), #statement_of_positive_emotion_pred
    ("statement_of_progress",         "statement_of_progress_pred"),
    ("statement_of_challenge",        "statement_of_challenge_pred"),
]


def f1_from_binary(y_true, y_pred):
    """
    Compute F1 = 2TP / (2TP + FP + FN) for binary {0,1} arrays.
    """
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    denom = (2 * tp + fp + fn)
    return (2.0 * tp / denom) if denom > 0 else 0.0

def bca_interval(theta_obs, theta_boot, jackknife_vals, alpha=0.05):
    """
    BCa interval from:
      - theta_obs: statistic on full data (float)
      - theta_boot: array of bootstrap statistics shape (B,)
      - jackknife_vals: array of jackknife leave-one-out statistics shape (n,)
      - alpha: two-sided alpha (e.g., 0.05)
    Returns (lower, upper).
    """
    theta_boot = np.asarray(theta_boot)
    B = len(theta_boot)
    prop_less = (np.sum(theta_boot < theta_obs) + 0.5 * np.sum(theta_boot == theta_obs)) / B
    prop_less = min(max(prop_less, 1e-10), 1 - 1e-10)
    z0 = scipy_stats_norm_ppf(prop_less)

    # Acceleration a via jackknife influence values
    jack = np.asarray(jackknife_vals)
    jack_mean = np.mean(jack)
    num = np.sum((jack_mean - jack) ** 3)
    den = 6.0 * (np.sum((jack_mean - jack) ** 2) ** 1.5)
    a = (num / den) if den > 0 else 0.0

    # Adjusted alpha levels
    lower_alpha = alpha / 2.0
    upper_alpha = 1.0 - alpha / 2.0

    def adj_alpha(raw_alpha):
        z = scipy_stats_norm_ppf(raw_alpha)
        denom = (1.0 - a * (z0 + z))
        if np.isclose(denom, 0.0):
            return 0.0 if (z0 + z) < 0 else 1.0
        adj = scipy_stats_norm_cdf(z0 + (z0 + z) / denom)
        # Restrict to [0,1]
        return float(min(max(adj, 0.0), 1.0))

    a1 = adj_alpha(lower_alpha)
    a2 = adj_alpha(upper_alpha)

    # Percentile points of the bootstrap distribution at adjusted alphas
    lower = np.quantile(theta_boot, a1, method="linear")
    upper = np.quantile(theta_boot, a2, method="linear")
    return float(lower), float(upper)

def scipy_stats_norm_cdf(x):
    return 0.5 * (1.0 + np.math.erf(x / np.sqrt(2.0)))

def scipy_stats_norm_ppf(p):
    # Acklam's approximation for inverse normal CDF
    p = float(min(max(p, 1e-12), 1 - 1e-12))
    # Coefficients
    a = [-3.969683028665376e+01, 2.209460984245205e+02,
         -2.759285104469687e+02, 1.383577518672690e+02,
         -3.066479806614716e+01, 2.506628277459239e+00]
    b = [-5.447609879822406e+01, 1.615858368580409e+02,
         -1.556989798598866e+02, 6.680131188771972e+01,
         -1.328068155288572e+01]
    c = [-7.784894002430293e-03, -3.223964580411365e-01,
         -2.400758277161838e+00, -2.549732539343734e+00,
          4.374664141464968e+00,  2.938163982698783e+00]
    d = [7.784695709041462e-03,  3.224671290700398e-01,
         2.445134137142996e+00,  3.754408661907416e+00]
    plow = 0.02425
    phigh = 1 - plow
    if p < plow:
        q = np.sqrt(-2 * np.log(p))
        return (((((c[0]*q + c[1])*q + c[2])*q + c[3])*q + c[4])*q + c[5]) / \
               ((((d[0]*q + d[1])*q + d[2])*q + d[3])*q + 1)
    elif p > phigh:
        q = np.sqrt(-2 * np.log(1 - p))
        return -(((((c[0]*q + c[1])*q + c[2])*q + c[3])*q + c[4])*q + c[5]) / \
                 ((((d[0]*q + d[1])*q + d[2])*q + d[3])*q + 1)
    else:
        q = p - 0.5
        r = q*q
        return (((((a[0]*r + a[1])*r + a[2])*r + a[3])*r + a[4])*r + a[5]) * q / \
               (((((b[0]*r + b[1])*r + b[2])*r + b[3])*r + b[4])*r + 1)


# Main computation

cols = sorted(set([t for t,_ in pairs] + [p for _,p in pairs]))
pdf = result_df.select(*cols).toPandas()

rng = np.random.default_rng(RANDOM_SEED)
n = len(pdf)

results = []

for true_col, pred_col in pairs:
    y_true = pdf[true_col].to_numpy()
    y_pred = pdf[pred_col].to_numpy()

    y_true = (y_true > 0).astype(int)
    y_pred = (y_pred > 0).astype(int)

    # Point estimate
    theta_obs = f1_from_binary(y_true, y_pred)

    # Bootstrap replicates
    boot_vals = np.empty(B, dtype=float)
    for b in range(B):
        idx = rng.integers(0, n, size=n)        # sample indices with replacement
        boot_vals[b] = f1_from_binary(y_true[idx], y_pred[idx])

    # Jackknife values (leave-one-out)
    jack_vals = np.empty(n, dtype=float)
    idx_all = np.arange(n)
    for i in range(n):
        mask = idx_all != i
        jack_vals[i] = f1_from_binary(y_true[mask], y_pred[mask])

    # BCa interval
    lo, hi = bca_interval(theta_obs, boot_vals, jack_vals, alpha=ALPHA)

    results.append({
        "category": true_col,
        "pred_col": pred_col,
        "F1_point": round(theta_obs, 4),
        "BCa_95_lo": round(lo, 4),
        "BCa_95_hi": round(hi, 4),
        "B": B
    })

res_df = pd.DataFrame(results).sort_values("category")
print(res_df.to_string(index=False))
